In [9]:
%load_ext autoreload
%autoreload 2
import jax
import pgx
from pgx.experimental import auto_reset
from twentyfortyeight import *
import jax.numpy as jnp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
env = pgx.make("2048")
batch_size = 4096

init = jax.jit(jax.vmap(env.init))  # vectorize and JIT-compile
step = jax.jit(jax.vmap(auto_reset(env.step, env.init)))

key = jax.random.key(42)
key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, batch_size)

state = init(keys)  # vectorized states
key, subkey = jax.random.split(key)
# policy_network = MLP(496, 4, subkey)
key, subkey = jax.random.split(key)
# value_network = MLP(496, 1, subkey)
obs_wrapper = ToInt(FlattenObservation())
ppo_network = PPONetwork(496, 4, key, observation_wrapper=obs_wrapper)
key, subkey = jax.random.split(key)
policy_fn = make_policy_fn(ppo_network.policy_forward)
value_fn = make_value_fn(ppo_network.value_forward)
next_state, current_obs, traj = collect_trajectory(step, state, state.observation, policy_fn, key, 16)

In [10]:
ppo_loss(traj, current_obs, ppo_network, value_fn, 0.99, 0.95,
         0.2, 1, 1)

(Array(1016.0814, dtype=float32),
 {'value_loss': Array(1018.6681, dtype=float32),
  'entropy_loss': Array(-1.2779311, dtype=float32),
  'policy_loss': Array(-1.3087387, dtype=float32)})